In [0]:
# ── Configuration ──────────────────────────────────────────────
# Catalog widget — change the default to reuse this notebook for another project.
dbutils.widgets.text("CATALOG", "skyLedger_airlines")

CATALOG = dbutils.widgets.get("CATALOG")
print(f"Catalog parameter : {CATALOG}")

In [0]:
%sql
-- ── Catalog ─────────────────────────────────────────────────
-- IDENTIFIER(:CATALOG) safely binds the widget value as a SQL identifier.
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:CATALOG)
COMMENT 'SkyLedger_Airlines - global airline booking and revenue management platform';

In [0]:
# ── Schemas (medallion + ops) ──────────────────────────────────
SCHEMAS = {
    "bronze":  "Ingested source data preserving native source structure",
    "silver":  "Cleaned, standardized, and validated data",
    "gold":    "Business-ready tables, aggregates, and analytics",
}

for schema, comment in SCHEMAS.items():
    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:name) COMMENT :comment",
        args={"name": f"{CATALOG}.{schema}", "comment": comment},
    )
    print(f"✓ Schema: {CATALOG}.{schema}")

In [0]:
%sql
SHOW SCHEMAS IN skyLedger_airlines

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS skyLedger_airlines.bronze.raw

In [0]:
# ── Source Directories ────────────────────────────────────────
BASE_PATH = f"/Volumes/{CATALOG}/bronze/raw"

SOURCE_FOLDERS = [
    "passenger",
    "booking",
    "flight booking",
    "route",
    "airline route",
    "airline",
    "nation",
    "region"
]

for folder in SOURCE_FOLDERS:
    dbutils.fs.mkdirs(f"{BASE_PATH}/{folder}")
    print(f"✓ Directory: {BASE_PATH}/{folder}")

In [0]:
# Verify write access by creating a temporary test table
TEST_TABLE = f"{CATALOG}.bronze._write_access_test"

spark.sql(f"""
    CREATE TABLE {TEST_TABLE} (
        test_id INT
    ) USING DELTA
""")

In [0]:
spark.sql(f"DROP TABLE {TEST_TABLE}")

print("Write access verified successfully.")

In [0]:
# 6. Verify source CSV read access
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/skyledger_airlines/bronze/raw/airline/")
)

print("Source CSV read access verified successfully.")
print("Columns:", df.columns)
print("Sample records:")
display(df.limit(5))
